# Parameter Golf — TT+MLA+GQA Training

**Self-contained Colab notebook.**  Clones from GitHub, trains, quantizes.

Sub-16MB language model with structural compression:
- **TT** (Tensor Train): 8× parameter reduction on Q/O projections
- **MLA** (Multi-head Latent Attention): 32× KV cache reduction
- **GQA** (Grouped-Query Attention): 8:1 query-to-KV head ratio

| Resource | Requirement |
|----------|-------------|
| GPU | Colab Pro A100 40GB |
| Training time | ~2-4 hours |
| Final artifact | <2 MB compressed (<16 MB limit) |
| Parameters | 4,370,704 (~8.74 MB BF16) |

## 1. Install Dependencies

In [ ]:
!pip install -q torch transformers tokenizers datasets tqdm numpy\n\nimport torch\nprint(f\"PyTorch {torch.__version__}\")\nprint(f\"CUDA: {torch.cuda.is_available()}\")\nif torch.cuda.is_available():\n    print(f\"GPU: {torch.cuda.get_device_name(0)}\")\n    print(f\"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB\")

## 2. Clone from GitHub

In [ ]:
!git clone -b tt-mla-gqa-submission https://github.com/ironbyte-rgb/parameter-golf.git /content/parameter-golf

import os, sys
os.chdir('/content/parameter-golf')
sys.path.insert(0, '/content/parameter-golf')
print(f"Working directory: {os.getcwd()}")
!ls -la *.py

## 3. Quick Architecture Check

In [ ]:
from model import TTMLATransformer\nimport torch\n\nmodel = TTMLATransformer()\ntotal, _ = model.count_parameters()\nprint(f\"Parameters: {total:,}  ({total*2/1e6:.2f} MB BF16)\")\nprint(f\"Under 16MB: {total * 2 < 16_000_000}\")\n\ndevice = torch.device('cuda')\nmodel = model.to(device)\nx = torch.randint(0, 4096, (4, 512), device=device)\ny = torch.randint(0, 4096, (4, 512), device=device)\nwith torch.no_grad():\n    loss = model(x, return_loss=True, targets=y)\nprint(f\"Initial loss: {loss.item():.2f} (random ~ln(4096)=8.32)\")\ndel model, x, y; torch.cuda.empty_cache()\nprint(\"\\nModel OK!\")

## 4. Train Tokenizer

In [ ]:
from tokenizer_ import train_bpe_tokenizer, build_byte_luts\n\ntokenizer = train_bpe_tokenizer(\n    output_path='./tokenizer.json',\n    vocab_size=4096,\n    sample_size=100_000_000,\n)\nluts = build_byte_luts(tokenizer, vocab_size=4096)\nprint(f\"Vocab: {tokenizer.get_vocab_size()}, LUTs ready\")

## 5. Train Model

In [ ]:
%cd /content/parameter-golf

import os
# --- TUNE THESE ---
os.environ['BATCH_SIZE'] = '128'
os.environ['TRAIN_STEPS'] = '10000'    # ~3.3B tokens
os.environ['LR'] = '6e-3'
os.environ['WARMUP_STEPS'] = '500'
os.environ['DECAY_STEPS'] = '1000'
os.environ['EVAL_EVERY'] = '1000'
os.environ['CHECKPOINT_EVERY'] = '2000'
os.environ['MAX_VAL_TOKENS'] = '200000'
os.environ['OUTPUT_DIR'] = '/content/parameter-golf/output'

print(f"Batch: {os.environ['BATCH_SIZE']} | Steps: {os.environ['TRAIN_STEPS']} | LR: {os.environ['LR']}")
print(f"Tokens/step: {int(os.environ['BATCH_SIZE']) * 512:,}")
print(f"Total tokens: {int(os.environ['BATCH_SIZE']) * 512 * int(os.environ['TRAIN_STEPS']):,}")

from train_gpt import train, get_config
cfg = get_config()
train(cfg)

## 6. Results

In [ ]:
%cd /content/parameter-golf

import os
output_dir = '/content/parameter-golf/output'
print("Output files:")
for f in sorted(os.listdir(output_dir)):
    path = os.path.join(output_dir, f)
    size_mb = os.path.getsize(path) / 1e6
    print(f"  {f:40s}  {size_mb:.2f} MB")

ptz_path = os.path.join(output_dir, 'final_model.int8.ptz')
if os.path.exists(ptz_path):
    ptz_size = os.path.getsize(ptz_path)
    code_path = '/content/parameter-golf/train_gpt.py'
    code_size = os.path.getsize(code_path) if os.path.exists(code_path) else 0
    total = ptz_size + code_size
    print(f"\n  Compressed model: {ptz_size:,} bytes ({ptz_size/1e6:.2f} MB)")
    print(f"  Code:             {code_size:,} bytes ({code_size/1e6:.2f} MB)")
    print(f"  Total artifact:   {total:,} bytes ({total/1e6:.2f} MB)")
    print(f"  Under 16MB:       {total < 16_000_000}")